# Gold - dim_plant

This notebook loads `amperedrive_giga.gold.dim_plant` from `amperedrive_giga.silver.dim_manufacturing_units`, one row per plant (`plant_id`). Load pattern: Full Overwrite / SCD Type 1. A full-sync MERGE updates changed rows, inserts new ones, and deletes plants that are no longer in silver, so the target always mirrors the source. The surrogate key is `xxhash64(CAST(plant_id AS STRING))`.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS amperedrive_giga.gold.dim_plant (
  plant_sk          BIGINT         NOT NULL COMMENT 'PK - xxhash64(plant_id)',
  plant_id          INT            NOT NULL COMMENT 'NK',
  plant_name        STRING         NOT NULL,
  unit_type         STRING,
  region            STRING         NOT NULL,
  country           STRING         NOT NULL,
  scrap_cost_factor DECIMAL(38,18) NOT NULL,
  source_created_at TIMESTAMP      NOT NULL,
  source_updated_at TIMESTAMP      NOT NULL,
  gold_load_ts      TIMESTAMP      NOT NULL,
  CONSTRAINT pk_dim_plant PRIMARY KEY (plant_sk)
)
COMMENT 'Gold plant dimension (SCD Type 1) sourced from silver.dim_manufacturing_units'

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_src_dim_plant AS
SELECT
  xxhash64(CAST(plant_id AS STRING)) AS plant_sk,
  plant_id,
  TRIM(plant_name)                   AS plant_name,
  TRIM(unit_type)                    AS unit_type,
  UPPER(TRIM(region))                AS region,
  UPPER(TRIM(country))               AS country,
  scrap_cost_factor,
  created_at                         AS source_created_at,
  updated_at                         AS source_updated_at
FROM amperedrive_giga.silver.dim_manufacturing_units
WHERE plant_id IS NOT NULL
-- Safety de-dup: keep the latest version per natural key
QUALIFY ROW_NUMBER() OVER (PARTITION BY plant_id ORDER BY updated_at DESC NULLS LAST) = 1

In [0]:
%sql
MERGE INTO amperedrive_giga.gold.dim_plant AS t
USING vw_src_dim_plant AS s
ON t.plant_id = s.plant_id
WHEN MATCHED AND (
     NOT (t.plant_name        <=> s.plant_name)
  OR NOT (t.unit_type         <=> s.unit_type)
  OR NOT (t.region            <=> s.region)
  OR NOT (t.country           <=> s.country)
  OR NOT (t.scrap_cost_factor <=> s.scrap_cost_factor)
  OR NOT (t.source_created_at <=> s.source_created_at)
  OR NOT (t.source_updated_at <=> s.source_updated_at)
) THEN UPDATE SET
  t.plant_name        = s.plant_name,
  t.unit_type         = s.unit_type,
  t.region            = s.region,
  t.country           = s.country,
  t.scrap_cost_factor = s.scrap_cost_factor,
  t.source_created_at = s.source_created_at,
  t.source_updated_at = s.source_updated_at,
  t.gold_load_ts      = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
  plant_sk, plant_id, plant_name, unit_type, region, country, scrap_cost_factor,
  source_created_at, source_updated_at, gold_load_ts
) VALUES (
  s.plant_sk, s.plant_id, s.plant_name, s.unit_type, s.region, s.country, s.scrap_cost_factor,
  s.source_created_at, s.source_updated_at, CURRENT_TIMESTAMP()
)
-- Full-overwrite semantics: remove plants no longer present in silver
WHEN NOT MATCHED BY SOURCE THEN DELETE

In [0]:
from delta.tables import DeltaTable

source_table = "amperedrive_giga.silver.dim_manufacturing_units"
staged_view = "vw_src_dim_plant"
target_fqn = "amperedrive_giga.gold.dim_plant"

rows_extracted = spark.table(source_table).count()  # rows read from silver source
rows_loaded = spark.table(staged_view).count()      # rows staged for the MERGE

# MERGE metrics: _sqldf holds the result of the MERGE cell above; fall back to the latest MERGE in Delta history
try:
    m = _sqldf.first().asDict()
    rows_inserted, rows_updated = int(m["num_inserted_rows"]), int(m["num_updated_rows"])
except Exception:
    h = (DeltaTable.forName(spark, target_fqn).history(10)
         .filter("operation = 'MERGE'").orderBy("version", ascending=False).first())
    om = h["operationMetrics"] if h else {}
    rows_inserted = int(om.get("numTargetRowsInserted", 0))
    rows_updated = int(om.get("numTargetRowsUpdated", 0))

result = {
    "rows_extracted": str(rows_extracted),
    "rows_loaded": str(rows_loaded),
    "rows_inserted": str(rows_inserted),
    "rows_updated": str(rows_updated)
}

dbutils.notebook.exit(result)